# Chapter 9: More on Specification and Data Issues

Topics: functional form misspecification, measurement error, missing data, outliers, LAD estimation.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import OLSInfluence
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 9.1 Functional Form Misspecification (RESET Test)

In [ ]:
df = woo.dataWoo('hprice1')
model = smf.ols('price ~ lotsize + sqrft + bdrms', data=df).fit()

from statsmodels.stats.diagnostic import linear_reset
reset = linear_reset(model, power=2, use_f=True)
print(f"RESET F-stat: {reset.statistic:.4f}, p-value: {reset.pvalue:.4f}")

# Log model
model_log = smf.ols('np.log(price) ~ np.log(lotsize) + np.log(sqrft) + bdrms', data=df).fit()
reset_log = linear_reset(model_log, power=2, use_f=True)
print(f"\nLog model RESET: F-stat={reset_log.statistic:.4f}, p-value={reset_log.pvalue:.4f}")

## 9.4 Outlying Observations

In [ ]:
df = woo.dataWoo('rdchem')
model = smf.ols('rdintens ~ sales + I(sales**2) + profmarg', data=df).fit()

influence = OLSInfluence(model)
stud_resid = influence.resid_studentized_external
leverage   = influence.hat_matrix_diag
cooks_d    = influence.cooks_distance[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(leverage, stud_resid, alpha=0.6, color='steelblue')
axes[0].axhline(2,  color='red', linestyle='--', label='±2')
axes[0].axhline(-2, color='red', linestyle='--')
axes[0].set_xlabel('Leverage'); axes[0].set_ylabel('Studentized Residuals')
axes[0].set_title("Leverage vs Studentized Residuals")
axes[0].legend()

axes[1].stem(cooks_d, markerfmt='C1o', linefmt='C1-', basefmt='gray')
axes[1].axhline(4/len(df), color='red', linestyle='--', label=f"4/n = {4/len(df):.3f}")
axes[1].set_title("Cook's Distance"); axes[1].legend()

plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_outliers.png", dpi=150)
plt.show()

outliers = np.where(np.abs(stud_resid) > 2)[0]
print(f"Potential outliers (|stud resid| > 2): observations {outliers}")